# Nemotron 3 Super Model Walkthrough

The key features demonstrated here are:
- `enable_thinking: true`
- `reasoning_budget`
- `low_effort: true`
- streaming responses
- tool-call streaming (`delta.tool_calls`)

In [ ]:
%pip install -q openai

## 1) How to get an NVIDIA API key

Use this once, then keep your key private.

1. Sign in to NVIDIA's API platform (for most users, this is the NVIDIA Build / API Catalog experience).
2. Open your account or API key settings page.
3. Create a new API key and copy it immediately (many portals only show it once).
4. Store it in a local secret manager or your shell profile, not in source control.
5. If your organization manages keys centrally, ask your platform admin for the correct key scope and endpoint policy.

Security note: never paste your real key into committed notebooks. Use environment variables or runtime prompts.

## 2) Configure API key, endpoint, and model

This cell prompts for your API key using `getpass`, asks for endpoint/model with defaults, and initializes the OpenAI client used in later examples.

In [ ]:
import os
from getpass import getpass

from openai import OpenAI

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
NVIDIA_API_URL = f"{NVIDIA_BASE_URL}/chat/completions"
MODEL = "nvidia/nemotron-3-super-120b-a12b"

NVIDIA_API_KEY = getpass("NVIDIA API key: " ).strip()

os.environ["NVIDIA_BASE_URL"] = NVIDIA_BASE_URL
os.environ["NVIDIA_API_URL"] = NVIDIA_API_URL
os.environ["MODEL"] = MODEL

CLIENT = OpenAI(
    base_url=NVIDIA_BASE_URL,
    api_key=NVIDIA_API_KEY,
    default_headers={"NVCF-POLL-SECONDS": "1800"},
)
STREAM_TIMEOUT_SECONDS = 1800
STREAM_REASONING_COLOR = "\033[32m"
STREAM_TOOL_COLOR = "\033[38;5;208m"  # orange
STREAM_RESET_COLOR = "\033[0m"

print(f"Configured endpoint: {NVIDIA_API_URL}")
print(f"Configured model: {MODEL}")

NVIDIA API key: ··········
Configured endpoint: https://integrate.api.nvidia.com/v1/chat/completions
Configured model: private/nvidia/nemotron-3-super-120b-a12b


### Streaming Helpers (Run Once)

In [ ]:
import json
import re

TOOL_PROMPT = "Use the get_math_answer tool to compute 123+456. Return only the final answer."
MATH_TOOL_SPEC = {
    "type": "function",
    "function": {
        "name": "get_math_answer",
        "description": "Returns an exact arithmetic result",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "Math expression"}},
            "required": ["expression"],
        },
    },
}

FOLLOW_UP_MESSAGES = None


def get_math_answer(expression: str) -> str:
    """Safe evaluator for simple arithmetic (digits, +, -, *, /, parentheses, spaces)."""
    expr = re.sub(r"\s+", "", expression)
    if not re.match(r"^[\d+\-*/().]+$", expr):
        return "Error: only numbers and + - * / ( ) allowed"
    try:
        return str(eval(expr))
    except Exception as e:
        return f"Error: {e}"


def create_stream(messages, extra_body=None, tools=None, tool_choice="auto"):
    req = {
        "model": MODEL,
        "messages": messages,
        "max_tokens": 8192,
        "temperature": 1.0,
        "top_p": 0.95,
        "stream": True,
        "timeout": STREAM_TIMEOUT_SECONDS,
    }
    if extra_body is not None:
        req["extra_body"] = extra_body
    if tools is not None:
        req["tools"] = tools
        req["tool_choice"] = tool_choice
    return CLIENT.chat.completions.create(**req)


def stream_reasoning_content(response, show_tool_deltas=False):
    in_reasoning = False
    acc = {}  # index -> {"id", "name", "arguments"}

    for chunk in response:
        for choice in chunk.choices:
            delta = choice.delta
            reasoning = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
            if reasoning:
                if not in_reasoning:
                    print(STREAM_REASONING_COLOR, end="")
                    in_reasoning = True
                print(reasoning, end="", flush=True)

            content = getattr(delta, "content", None)
            if content:
                if in_reasoning:
                    print(STREAM_RESET_COLOR, end="")
                    in_reasoning = False
                print(content, end="", flush=True)

            if not show_tool_deltas:
                continue

            tool_calls = getattr(delta, "tool_calls", None)
            if not tool_calls:
                continue

            for tc in tool_calls:
                idx = tc.get("index", 0) if isinstance(tc, dict) else getattr(tc, "index", 0)
                idx = 0 if idx is None else idx
                if idx not in acc:
                    acc[idx] = {"id": "", "name": "", "arguments": ""}

                if isinstance(tc, dict):
                    acc[idx]["id"] = acc[idx]["id"] or tc.get("id", "")
                    fn = tc.get("function", {})
                    name_part = fn.get("name", "")
                    args_part = fn.get("arguments", "")
                else:
                    acc[idx]["id"] = acc[idx]["id"] or getattr(tc, "id", "")
                    fn = getattr(tc, "function", None)
                    name_part = (getattr(fn, "name", None) or "") if fn else ""
                    args_part = (getattr(fn, "arguments", None) or "") if fn else ""

                if name_part:
                    print(STREAM_TOOL_COLOR + f"\n[tool#{idx} name+] {name_part}" + STREAM_RESET_COLOR, end="", flush=True)
                if args_part:
                    print(STREAM_TOOL_COLOR + f"\n[tool#{idx} args+] {args_part}" + STREAM_RESET_COLOR, end="", flush=True)

                acc[idx]["name"] += name_part
                acc[idx]["arguments"] += args_part

    if in_reasoning:
        print(STREAM_RESET_COLOR, end="")
    print()
    return acc


def execute_local_tools(acc):
    assistant_tool_calls, tool_results = [], []
    if not acc:
        return assistant_tool_calls, tool_results

    print(STREAM_TOOL_COLOR + "Tool call(s) and result(s):" + STREAM_RESET_COLOR)
    for idx in sorted(acc.keys()):
        tc = acc[idx]
        name, args_str = tc["name"], tc["arguments"]
        if not name:
            continue

        try:
            args = json.loads(args_str) if args_str.strip() else {}
        except json.JSONDecodeError:
            args = {}

        result = get_math_answer(args.get("expression", "")) if name == "get_math_answer" else f"(unknown tool: {name})"
        print(STREAM_TOOL_COLOR + f"  {name}({args}) -> {result}" + STREAM_RESET_COLOR)

        assistant_tool_calls.append({"id": tc["id"], "type": "function", "function": {"name": name, "arguments": args_str}})
        tool_results.append({"role": "tool", "tool_call_id": tc["id"], "content": result})

    return assistant_tool_calls, tool_results

## 3) `enable_thinking: true` (no budget)

This asks the model to emit reasoning tokens first in the stream, then user-facing content.

Rendered output behavior in this notebook:
- reasoning deltas print first in green
- response content prints after in default terminal color

In [ ]:
response = create_stream(
    messages=[{"role": "user", "content": "What is 2+2?"}],
)
stream_reasoning_content(response)

The user asks a simple arithmetic: 2+2=4. Provide answer.

2 + 2 = 4.


{}

<details>
<summary>Equivalent curl command</summary>

```bash
NVIDIA_API_KEY="<your_nvidia_api_key>"
NVIDIA_API_URL="https://integrate.api.nvidia.com/v1/chat/completions"
MODEL="private/nvidia/nemotron-3-super-120b-a12b"

curl -sS -N "$NVIDIA_API_URL" \
  -H "Authorization: Bearer $NVIDIA_API_KEY" \
  -H "Content-Type: application/json" \
  -H "NVCF-POLL-SECONDS: 1800" \
  -d '{
    "model": "'"$MODEL"'",
    "messages": [{"role": "user", "content": "What is 2+2?"}],
    "max_tokens": 8192,
    "temperature": 1.0,
    "top_p": 0.95,
    "chat_template_kwargs": {"enable_thinking": true},
    "stream": true
  }'
```

</details>

## 4) `enable_thinking: true` with `reasoning_budget`

A reasoning budget gives the model room to think longer before final output.
In practice, you should expect more `delta.reasoning` tokens compared to no budget.

In [ ]:
response = create_stream(
    messages=[{"role": "user", "content": "What is the captial of France?"}],
    extra_body={
        "reasoning_budget": 8192,
        "chat_template_kwargs": {"enable_thinking": True},
    },
)
stream_reasoning_content(response)

Okay, the user is asking for the capital of France. This seems like a very basic geography question, probably from someone learning about world capitals or maybe a young student. 

Hmm, the spelling "captial" is clearly a typo—they meant "capital." But since the question is so fundamental, I shouldn't overcomplicate it. They likely just need a quick, accurate answer to move on with whatever they're doing. 

I recall Paris is the capital—no controversy there, unlike some disputed capitals. Better keep it simple: just state the answer clearly. No need for extra fluff unless they ask for more details. 

...Though part of me wants to add that it's also the largest city and a cultural hub, but that might be overkill for such a straightforward query. Stick to the point unless they follow up. 

*Double-checks mentally*: Yep, definitely Paris. Not London, not Berlin—common mix-ups but irrelevant here. 

Answer locked in: Paris.

The capital of France is **Paris**.  This is a well-established f

{}

<details>
<summary>Equivalent curl command</summary>

```bash
NVIDIA_API_KEY="<your_nvidia_api_key>"
NVIDIA_API_URL="https://integrate.api.nvidia.com/v1/chat/completions"
MODEL="private/nvidia/nemotron-3-super-120b-a12b"

curl -sS -N "$NVIDIA_API_URL" \
  -H "Authorization: Bearer $NVIDIA_API_KEY" \
  -H "Content-Type: application/json" \
  -H "NVCF-POLL-SECONDS: 1800" \
  -d '{
    "model": "'"$MODEL"'",
    "messages": [{"role": "user", "content": "What is 2+2? Think step by step."}],
    "max_tokens": 8192,
    "temperature": 1.0,
    "top_p": 0.95,
    "reasoning_budget": 8192,
    "chat_template_kwargs": {"enable_thinking": true},
    "stream": true
  }'
```

</details>

## 5) `enable_thinking: true` + `low_effort: true`

`low_effort` is useful when latency and cost matter more than deeper reasoning depth.
You usually get a shorter reasoning sequence plus concise final output.

In [ ]:
response = create_stream(
    messages=[{"role": "user", "content": "What is NVIDIA?"}],
    extra_body={"chat_template_kwargs": {"enable_thinking": True, "low_effort": True}},
)
stream_reasoning_content(response)

User asks: What is NVIDIA? Simple answer.

NVIDIA is an American multinational technology company best known for designing and manufacturing graphics processing units (GPUs) and related hardware and software for gaming, professional visualization, data centers, artificial intelligence (AI), and automotive applications. Founded in 1993 by Jensen Huang, Chris Malachowsky, and Curtis Priem, the company’s innovations—such as the GeForce series for gamers, the Quadro and RTX lines for creators and professionals, and the Tesla and A100/H100 series for AI and high‑performance computing—have made it a cornerstone of modern computing. NVIDIA also develops the CUDA parallel computing platform, AI frameworks like TensorRT and cuDNN, and software ecosystems (e.g., Omniverse, Drive, and Isaac) that enable developers to leverage its GPU architecture for a wide range of workloads, from real‑time ray tracing to deep‑learning research and autonomous‑vehicle systems.


{}

<details>
<summary>Equivalent curl command</summary>

```bash
NVIDIA_API_KEY="<your_nvidia_api_key>"
NVIDIA_API_URL="https://integrate.api.nvidia.com/v1/chat/completions"
MODEL="private/nvidia/nemotron-3-super-120b-a12b"

curl -sS -N "$NVIDIA_API_URL" \
  -H "Authorization: Bearer $NVIDIA_API_KEY" \
  -H "Content-Type: application/json" \
  -H "NVCF-POLL-SECONDS: 1800" \
  -d '{
    "model": "'"$MODEL"'",
    "messages": [{"role": "user", "content": "What is 2+2?"}],
    "max_tokens": 8192,
    "temperature": 1.0,
    "top_p": 0.95,
    "chat_template_kwargs": {"enable_thinking": true, "low_effort": true},
    "stream": true
  }'
```

</details>

## 6) Tool call + streaming + thinking (no budget)

This demonstrates function-calling behavior in stream mode.

Expected flow:
- reasoning deltas
- `delta.tool_calls` with function name and incremental JSON arguments
- final chunk with `finish_reason: "tool_calls"`, then `[DONE]`

### Tool-Call Demo

In [ ]:
# Run tool-call demo (initial stream + local tool execution)
user_msg = {"role": "user", "content": TOOL_PROMPT}
response = create_stream(
    messages=[user_msg],
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    tools=[MATH_TOOL_SPEC],
)
acc = stream_reasoning_content(response, show_tool_deltas=True)
assistant_tool_calls, tool_results = execute_local_tools(acc)

FOLLOW_UP_MESSAGES = None
if assistant_tool_calls:
    FOLLOW_UP_MESSAGES = [
        user_msg,
        {"role": "assistant", "content": None, "tool_calls": assistant_tool_calls},
        *tool_results,
    ]
    print("Tools executed in this cell. Run the next cell for the final assistant response.")
else:
    print("No tool calls were emitted in the initial stream.")

We need to compute 123+456 using get_math_answer. The tool returns exact arithmetic result. So we call get_math_answer with expression "123+456". Then return only final answer. Let's do that.


[tool#0 name+] get_math_answer
[tool#0 args+] {
[tool#0 args+] "expression": "123+456"
[tool#0 args+] }
Tool call(s) and result(s):
  get_math_answer({'expression': '123+456'}) -> 579
Tools executed in this cell. Run the next cell for the final assistant response.


In [ ]:
# Final assistant response after tool execution in previous cell
if not FOLLOW_UP_MESSAGES:
    print("No prepared follow-up messages. Re-run the previous cell first.")
else:
    final_stream = create_stream(
        messages=FOLLOW_UP_MESSAGES,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}},
        tools=[MATH_TOOL_SPEC],
    )
    stream_reasoning_content(final_stream)

We need to output only the final answer: 579. The instruction: "Return only the final answer." So just output 579.



579


<details>
<summary>Equivalent curl command</summary>

```bash
NVIDIA_API_KEY="<your_nvidia_api_key>"
NVIDIA_API_URL="https://integrate.api.nvidia.com/v1/chat/completions"
MODEL="private/nvidia/nemotron-3-super-120b-a12b"

curl -sS -N "$NVIDIA_API_URL" \
  -H "Authorization: Bearer $NVIDIA_API_KEY" \
  -H "Content-Type: application/json" \
  -H "NVCF-POLL-SECONDS: 1800" \
  -d '{
    "model": "'"$MODEL"'",
    "messages": [{"role": "user", "content": "Use the get_math_answer tool to compute 123+456. Return only the final answer."}],
    "max_tokens": 8192,
    "temperature": 1.0,
    "top_p": 0.95,
    "chat_template_kwargs": {"enable_thinking": true},
    "tools": [
      {
        "type": "function",
        "function": {
          "name": "get_math_answer",
          "description": "Returns an exact arithmetic result",
          "parameters": {
            "type": "object",
            "properties": {
              "expression": {"type": "string", "description": "Math expression"}
            },
            "required": ["expression"]
          }
        }
      }
    ],
    "tool_choice": "auto",
    "stream": true
  }'
```

</details>

### 7) Use Perplexity Search - Powered by Nemotron 3 Super!

In [ ]:
import requests
import json
from getpass import getpass

api_key = getpass("Enter Perplexity API key: ")

Enter Perplexity API key: ··········


In [ ]:
response = requests.post(
    "https://api.perplexity.ai/v1/responses",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "input": "What new architecture is in the Nemotron 3 family of models?",
        "tools": [{"type": "web_search"}],
        "instructions": "You have access to a web_search tool. Use it for questions about current events, news, or recent developments.",
    }
)

data = response.json()
for block in data["output"]:
    if block.get("type") == "message":
        for content in block["content"]:
            if content.get("type") == "output_text":
                print(content["text"])


The Nemotron 3 family introduces a **hybrid Mamba‑Transformer Mixture‑of‑Experts (MoE) architecture** as its core innovation.  

Key points from the recent announcements and technical details:

| Aspect | Description |
|--------|-------------|
| **Base design** | A backbone that interleaves **Mamba‑2 (state‑space) layers** with **Transformer attention** layers, giving the model the speed and long‑context strengths of Mamba while retaining the expressive power of Transformers. |
| **Mixture‑of‑Experts** | Sparse MoE layers replace the traditional feed‑forward networks. A learned router sends each token to only a small subset of expert networks (e.g., Nano activates ~3 B of its ~30 B parameters per token). This yields high throughput and low inference cost while preserving a large effective parameter count. |
| **Latent MoE (Super & Ultra)** | A hardware‑aware “LatentMoE” variant that further improves expert routing and load balancing, boosting accuracy and efficiency for the larger mod